In [2]:
# ==============================================================================
# [1] 필수 패키지 설치 및 환경 설정
# ==============================================================================
!pip -q install catboost==1.2.8 lightgbm

import os
import gc
import glob
import time
import joblib
import shutil
import zipfile
import subprocess
import sys
import torch
import numpy as np
import pandas as pd
from pathlib import Path

import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss
from google.colab import files

# 경로 및 디렉터리 설정
if Path('/content/train.csv').exists():
    DATA_DIR = Path('/content')
elif Path('/content/data/train.csv').exists():
    DATA_DIR = Path('/content/data')
else:
    DATA_DIR = Path('.')

WORK_DIR = Path('/content/work')
MODEL_DIR = WORK_DIR / 'model'
ZIP_PATH = Path('/content/submit.zip')

MODEL_DIR.mkdir(parents=True, exist_ok=True)

ID_COL = "row_id"
TARGET_COL = "control_success"
TRAIN_PATH = DATA_DIR / 'train.csv'

assert TRAIN_PATH.exists(), f"❌ {TRAIN_PATH} 파일이 없습니다!"

print(f"1. 데이터 로드 중... ({TRAIN_PATH})")
df_train = pd.read_csv(TRAIN_PATH, encoding="utf-8-sig")

# ------------------------------------------------------------------------------
# 파생변수 생성 함수 (볼카운트 + hand_match + 접전 + 득점권 상호작용)
# ------------------------------------------------------------------------------
def add_all_features(df):
    df_out = df.copy()

    # 1. 볼카운트 피처 (4종)
    ball_col = [c for c in df_out.columns if c.lower() in ['ball', 'balls', 'b']][0] if any(c.lower() in ['ball', 'balls', 'b'] for c in df_out.columns) else None
    strike_col = [c for c in df_out.columns if c.lower() in ['strike', 'strikes', 's']][0] if any(c.lower() in ['strike', 'strikes', 's'] for c in df_out.columns) else None

    if ball_col and strike_col:
        df_out['ball_strike_diff'] = (df_out[ball_col] - df_out[strike_col]).astype('float32')
        is_hitter = ((df_out[ball_col] == 2) & (df_out[strike_col] == 0)) | \
                    ((df_out[ball_col] == 3) & (df_out[strike_col] == 0)) | \
                    ((df_out[ball_col] == 3) & (df_out[strike_col] == 1))
        df_out['is_hitter_count'] = is_hitter.astype('float32')
        df_out['is_2strike'] = (df_out[strike_col] == 2).astype('float32')
        df_out['is_full_count'] = ((df_out[ball_col] == 3) & (df_out[strike_col] == 2)).astype('float32')

    # 2. 손잡이 상성 (hand_match)
    if 'p_throws' in df_out.columns and 'b_bats' in df_out.columns:
        df_out['hand_match'] = df_out['p_throws'].astype(str) + "_" + df_out['b_bats'].astype(str)

    # 3. 접전 상황 지표
    score_cols = [c for c in df_out.columns if 'score' in c.lower() or 'diff' in c.lower()]
    if len(score_cols) >= 2:
        try:
            diff = abs(df_out[score_cols[0]] - df_out[score_cols[1]])
            df_out['abs_score_diff'] = diff.astype('float32')
            df_out['is_close_game'] = (diff <= 2).astype('float32')
        except:
            pass

    # 4. 득점권 지표 및 상호작용
    if 'base_state' in df_out.columns:
        b_str = df_out['base_state'].astype(str).str.replace(" ", "")
        has_2b_or_3b = b_str.str.contains('2') | b_str.str.contains('3')
        df_out['is_scoring_position'] = has_2b_or_3b.astype('float32')

        if 'is_hitter_count' in df_out.columns:
            df_out['scoring_hitter_count_risk'] = (df_out['is_scoring_position'] * df_out['is_hitter_count']).astype('float32')

    return df_out

print("2. 파생변수 생성 중...")
df_train = add_all_features(df_train)

# 메모리 절약형 float32 캐스팅
for col in df_train.select_dtypes(include=['float64']).columns:
    df_train[col] = df_train[col].astype('float32')

# 범주형 컬럼 탐색 및 category 타입 변환
cat_cols = ["top_bottom", "game_type", "base_state", "hand_match", "p_throws", "b_bats", "pitch_type"]
cat_cols = [c for c in cat_cols if c in df_train.columns]

for col in cat_cols:
    df_train[col] = df_train[col].astype("category")

features = [col for col in df_train.columns if col not in [ID_COL, TARGET_COL]]
X = df_train[features].copy()
y = df_train[TARGET_COL].values.astype(np.int8)

use_gpu = torch.cuda.is_available()

# ==============================================================================
# [2] TargetEncoder 없이 모델 내장 기능으로 5-Fold 학습
# ==============================================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(df_train), dtype=np.float32)
oof_cat = np.zeros(len(df_train), dtype=np.float32)

print(f"\n🚀 5-Fold 내장 범주형 학습 시작... (하드웨어: {'GPU' if use_gpu else 'CPU'})")
total_t0 = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n==================== Fold {fold + 1} / 5 시작 ====================")
    X_tr, y_tr = X.iloc[train_idx].copy(), y[train_idx]
    X_va, y_va = X.iloc[val_idx].copy(), y[val_idx]

    # 1) LightGBM (내장 category 파라미터 자동 처리)
    model_lgb = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.08, max_depth=8, num_leaves=31,
        random_state=42, n_jobs=4
    )
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(period=100)
        ]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va)[:, 1].astype(np.float32)

    # 2) CatBoost (cat_features 직접 지정)
    model_cat = CatBoostClassifier(
        iterations=1000, learning_rate=0.08, depth=6, random_seed=42,
        verbose=100, thread_count=4, task_type="GPU" if use_gpu else "CPU",
        cat_features=cat_cols
    )
    model_cat.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        early_stopping_rounds=30
    )
    oof_cat[val_idx] = model_cat.predict_proba(X_va)[:, 1].astype(np.float32)

    model_lgb.booster_.save_model(str(MODEL_DIR / f"model_lgb_fold{fold}.txt"))
    model_cat.save_model(str(MODEL_DIR / f"model_cat_fold{fold}.cbm"))

    del X_tr, X_va
    gc.collect()

oof_final = 0.5 * oof_lgb + 0.5 * oof_cat
print(f"\n 전체 학습 완료 (총 소요 시간: {(time.time() - total_t0)/60:.1f}분)")
print(f"5-Fold OOF AUC 점수: {roc_auc_score(y, oof_final):.5f}")
print(f"5-Fold OOF Brier Score: {brier_score_loss(y, oof_final):.5f}")

# ==============================================================================
# [3] script.py 동기화 및 검증
# ==============================================================================
SCRIPT_CONTENT = r'''import os
import glob
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from pathlib import Path

ID_COL = "row_id"
TARGET_COL = "control_success"

def add_all_features(df):
    df_out = df.copy()
    ball_col = [c for c in df_out.columns if c.lower() in ['ball', 'balls', 'b']][0] if any(c.lower() in ['ball', 'balls', 'b'] for c in df_out.columns) else None
    strike_col = [c for c in df_out.columns if c.lower() in ['strike', 'strikes', 's']][0] if any(c.lower() in ['strike', 'strikes', 's'] for c in df_out.columns) else None

    if ball_col and strike_col:
        df_out['ball_strike_diff'] = (df_out[ball_col] - df_out[strike_col]).astype('float32')
        is_hitter = ((df_out[ball_col] == 2) & (df_out[strike_col] == 0)) | \
                    ((df_out[ball_col] == 3) & (df_out[strike_col] == 0)) | \
                    ((df_out[ball_col] == 3) & (df_out[strike_col] == 1))
        df_out['is_hitter_count'] = is_hitter.astype('float32')
        df_out['is_2strike'] = (df_out[strike_col] == 2).astype('float32')
        df_out['is_full_count'] = ((df_out[ball_col] == 3) & (df_out[strike_col] == 2)).astype('float32')

    if 'p_throws' in df_out.columns and 'b_bats' in df_out.columns:
        df_out['hand_match'] = df_out['p_throws'].astype(str) + "_" + df_out['b_bats'].astype(str)

    score_cols = [c for c in df_out.columns if 'score' in c.lower() or 'diff' in c.lower()]
    if len(score_cols) >= 2:
        try:
            diff = abs(df_out[score_cols[0]] - df_out[score_cols[1]])
            df_out['abs_score_diff'] = diff.astype('float32')
            df_out['is_close_game'] = (diff <= 2).astype('float32')
        except:
            pass

    if 'base_state' in df_out.columns:
        b_str = df_out['base_state'].astype(str).str.replace(" ", "")
        has_2b_or_3b = b_str.str.contains('2') | b_str.str.contains('3')
        df_out['is_scoring_position'] = has_2b_or_3b.astype('float32')
        if 'is_hitter_count' in df_out.columns:
            df_out['scoring_hitter_count_risk'] = (df_out['is_scoring_position'] * df_out['is_hitter_count']).astype('float32')

    return df_out

def main():
    root = Path(__file__).resolve().parent
    test_path = root / "data" / "test.csv"
    sample_sub_path = root / "data" / "sample_submission.csv"
    out_path = root / "output" / "submission.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if not test_path.exists():
        test_path = Path("./data/test.csv")
    if not sample_sub_path.exists():
        sample_sub_path = Path("./data/sample_submission.csv")

    df_test = pd.read_csv(test_path, encoding="utf-8-sig")
    df_sub = pd.read_csv(sample_sub_path, encoding="utf-8-sig")

    df_test = add_all_features(df_test)

    for col in df_test.select_dtypes(include=['float64']).columns:
        df_test[col] = df_test[col].astype('float32')

    cat_cols = ["top_bottom", "game_type", "base_state", "hand_match", "p_throws", "b_bats", "pitch_type"]
    for col in cat_cols:
        if col in df_test.columns:
            df_test[col] = df_test[col].astype("category")

    drop_cols = [c for c in [ID_COL, TARGET_COL] if c in df_test.columns]
    X_test = df_test.drop(columns=drop_cols).copy()

    model_dir = root / "model"
    lgb_files = sorted(glob.glob(str(model_dir / "model_lgb_fold*.txt")))
    cat_files = sorted(glob.glob(str(model_dir / "model_cat_fold*.cbm")))

    lgb_preds, cat_preds = [], []

    for lgb_f, cat_f in zip(lgb_files, cat_files):
        # LightGBM 예측
        lgb_m = lgb.Booster(model_file=lgb_f)
        pred_lgb = lgb_m.predict(X_test)
        if pred_lgb.ndim > 1:
            pred_lgb = pred_lgb[:, 1]
        lgb_preds.append(pred_lgb)

        # CatBoost 예측
        cb_m = CatBoostClassifier()
        cb_m.load_model(cat_f)
        pred_cat = cb_m.predict_proba(X_test)[:, 1]
        cat_preds.append(pred_cat)

    mean_lgb = np.mean(lgb_preds, axis=0)
    mean_cat = np.mean(cat_preds, axis=0)
    final_preds = np.clip(0.5 * mean_lgb + 0.5 * mean_cat, 0.0, 1.0)

    pred_map = dict(zip(df_test[ID_COL].astype(str), final_preds))
    df_sub[TARGET_COL] = df_sub[ID_COL].astype(str).map(pred_map).fillna(0.5)

    df_sub.to_csv(out_path, index=False, encoding="utf-8-sig")
    print("Inference Finished Successfully!")

if __name__ == "__main__":
    main()
'''

(WORK_DIR / 'script.py').write_text(SCRIPT_CONTENT, encoding='utf-8')
(WORK_DIR / 'requirements.txt').write_text("catboost==1.2.8\nlightgbm\n", encoding='utf-8')

# ==============================================================================
# [4] 가상 테스트 구동 및 submit.zip 압축/다운로드
# ==============================================================================
print("\n3. script.py 사전 가상 테스트 진행 중...")
package_data = WORK_DIR / 'data'
package_output = WORK_DIR / 'output'
package_data.mkdir(exist_ok=True); package_output.mkdir(exist_ok=True)

train_sample = df_train.head(5).copy()
train_sample.drop(columns=[TARGET_COL], errors='ignore').to_csv(package_data / 'test.csv', index=False, encoding='utf-8-sig')
train_sample[[ID_COL, TARGET_COL]].to_csv(package_data / 'sample_submission.csv', index=False, encoding='utf-8-sig')

result = subprocess.run([sys.executable, str(WORK_DIR / 'script.py')], cwd=WORK_DIR, capture_output=True, text=True)

if result.returncode != 0:
    print("script.py 사전 테스트 실행 오류!")
    print(result.stderr)
    raise RuntimeError("검증 실패")
else:
    print("script.py 사전 테스트 성공!")

shutil.rmtree(package_data)
shutil.rmtree(package_output)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(WORK_DIR / 'script.py', arcname='script.py')
    zipf.write(WORK_DIR / 'requirements.txt', arcname='requirements.txt')

    for file in MODEL_DIR.rglob('*'):
        if file.is_file():
            arcname = file.relative_to(WORK_DIR)
            zipf.write(file, arcname=arcname)

print("\n TargetEncoder 제거 버전 submit.zip 작성이 완료되었습니다!")
files.download(ZIP_PATH)

1. 데이터 로드 중... (/content/train.csv)
2. 파생변수 생성 중...

🚀 5-Fold 내장 범주형 학습 시작... (하드웨어: CPU)

==================== Fold 1 / 5 시작 ====================
[LightGBM] [Info] Number of positive: 618082, number of negative: 561991
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.240820 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6418
[LightGBM] [Info] Number of data points in the train set: 1180073, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.523766 -> initscore=0.095135
[LightGBM] [Info] Start training from score 0.095135
[100]	valid_0's binary_logloss: 0.683034
[200]	valid_0's binary_logloss: 0.68257
[300]	valid_0's binary_logloss: 0.68232
[400]	valid_0's binary_logloss: 0.682157
[500]	valid_0's binary_logloss: 0.682037
[600]	valid_0's binary_logloss: 0.681943
[700]	valid_0's binary_logloss: 0.681894
0:	learn

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>